# Phase 6: Fast MoCo-v2 Pretraining on Google Colab

**Optimized for Colab Free Tier (T4 GPU)**:
- **Mixed Precision (AMP FP16)**: ~2.5x speedup using T4 Tensor Cores.
- **Optimized Evaluation & FFT Front-End**: Bypasses redundant transforms and evaluates k-NN every 5 epochs.
- **Epoch Budget (60 epochs)**: MoCo converges in 50-60 epochs on DeepShip acoustics (~10-12 mins per model vs 2+ hours).
- **Live Google Drive Checkpoint Sync**: Checkpoints save directly to Drive after every single epoch. If Colab disconnects, you can resume without losing any progress using `--resume`.

Make sure your runtime is set to **GPU (T4)**: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Mount Google Drive to persist all checkpoints and logs
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/deepsonar_v2_checkpoints"
!mkdir -p "/content/drive/MyDrive/deepsonar_v2_results"

In [ ]:
# 3. Clone repository and install dependencies
%cd /content
!rm -rf deepsonar_v2
!git clone https://github.com/Adhi-1004/deepsonar_v2.git
%cd /content/deepsonar_v2
!bash scripts/00_setup_env.sh

# Symlink checkpoints folder to Google Drive so EVERY epoch is saved immediately to Drive
!mkdir -p checkpoints
!rm -rf checkpoints
!ln -s "/content/drive/MyDrive/deepsonar_v2_checkpoints" checkpoints
print("Checkpoints live-linked to Google Drive!")

In [ ]:
# 4. Set Kaggle credentials & Login to W&B (optional)
import os
import wandb

# Enter your Kaggle API key here:
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USERNAME"
os.environ["KAGGLE_KEY"] = "YOUR_KAGGLE_KEY"

# Optional: login to Weights & Biases (or leave offline)
# wandb.login()

In [ ]:
# 5. Download DeepShip & build cache
%cd /content/deepsonar_v2
!bash scripts/01_fetch_data.sh
!python scripts/02_build_manifests.py
!python scripts/03_build_splits.py
!python scripts/04_clean_audio.py
!python scripts/04_segment_and_cache.py --dataset deepship
# Also build the logmel feature cache for downstream fast eval
!python scripts/04_segment_and_cache.py --dataset deepship --features logmel

## Run Pretraining (Fast, ~10-12 mins each on T4)

The models run with `--amp` (Automatic Mixed Precision), batch size 128, and 60 epochs.
If your session gets disconnected, simply re-run the cell with `--resume` to continue exactly where it left off.

In [ ]:
# Variant 1: Physics-Augmented MoCo (Headline model - Run this first!)
# Takes ~10-12 minutes for 60 epochs
%cd /content/deepsonar_v2
!python scripts/05_pretrain.py \
    --config configs/pretrain/moco_physics.yaml \
    --epochs 60 \
    --batch-size 128 \
    --workers 2 \
    --amp \
    --monitor-every 5


In [ ]:
# Variant 2: Generic Baseline MoCo (Ablation model)
# Takes ~10 minutes for 60 epochs
%cd /content/deepsonar_v2
!python scripts/05_pretrain.py \
    --config configs/pretrain/moco_generic.yaml \
    --epochs 60 \
    --batch-size 128 \
    --workers 2 \
    --amp \
    --monitor-every 5


In [ ]:
# Variant 3: Physics-Augmented with DEMON Consistency
# Takes ~12-14 minutes for 60 epochs
%cd /content/deepsonar_v2
!python scripts/05_pretrain.py \
    --config configs/pretrain/moco_physics_demon.yaml \
    --epochs 60 \
    --batch-size 128 \
    --workers 2 \
    --amp \
    --monitor-every 5


In [ ]:
# Checkpoint verification
!ls -lh /content/drive/MyDrive/deepsonar_v2_checkpoints/*